In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline


In [2]:
import torch
import torch.nn as nn   
import torch.optim as optim

from torch.optim import SGD

from torch.utils.data import DataLoader, TensorDataset

In [10]:
import os

In [4]:
comet_key = os.environ.get("comet_ml_key")

In [5]:
import comet_ml
# comet_ml.login(comet_key)

In [6]:
exp = comet_ml.start(api_key = comet_key, project_name = "Dissertation ML")

COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/alcatraz312/dissertation-ml/700220d2e8c243ff90b940dbcf6f2eaf



## The Model


### Variational Autoencoder

In [ ]:
class MTLArchitecture(nn.Module):
    def __init__(self, input_dim, latent_dim, num_classes):
        '''
        input_dim : number of wavelength pixels in the spectrum  -> int \n
        latent_dim : dimension of the latent space  -> int \n
        num_classes : number of stellar spectral classes  -> int
        '''
        super().__init__()

        # Encoder structure
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
        )

        # latent distribution
        self.mu_layer = nn.Linear(128, latent_dim)   # mean latent vector 
        self.logvar_layer = nn.Linear(128, latent_dim)  # variance latent vector

        # Decoder structure
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Linear(1024, input_dim)
        )

        # Downstream tasks
        # Regression head
        self.regression_head = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU()
        )

        self.reg_mu = nn.Linear(32, 3)    # estimated atmospheric parameters vector
        self.reg_logvar = nn.Linear(32,3)    # estimation uncertainties

        # Classification head
        self.classification_head = nn.Sequential(
                nn.Linear(latent_dim, 64),
                nn.ReLU(),
                nn.Linear(64,num_classes)
        )

    def reparameterization(self, mu, logvar):           # reparameterization for sampling 
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)          # sample from normal distribution

        return mu + eps * std

    # Feed forward
    def forward(self, x):
        h = self.encoder(x)      # encoded data

        mu = self.mu_layer(h)    # encoded mean
        logvar = self.logvar_layer(h)    # encoded uncertainty 

        z = self.reparameterization(mu, logvar)    # reparameterized latent variable 

        z_reg = self.regression_head(z)    

        mu_reg = self.reg_mu(z_reg)             # estimated atmospheric parameters
        logvar_reg = self.reg_logvar(z_reg)         # log variance of the atmospheric parameters
 
        cls_logits = self.classification_head(z)      # raw classification logits

        x_hat = self.decoder(z)       # reconstructed data

        out_data = {"x reconstructed" : x_hat,
                    "encoded mean" : mu,
                    "encoded log variance" : logvar,
                    "estimated atmospheric parameters": mu_reg,
                    "estimated parameters uncertainty": logvar_reg,
                    "class logits" : cls_logits}
        
        return out_data
    
    # Gaussian reconstruction NLL
    def gaussian_loglikelihood(self, x, x_hat, logvar):
        '''
        VAE reconstruction loss function \n
        Parameters: \n
        x : input spectrum  -> tensor \n
        x_hat : reconstructed spectrum  -> tensor \n
        logvar : 
        '''

        logvar = torch.clamp(logvar, -10, 5)
        
        return 0.5 * torch.sum(
            logvar + (x - x_hat)**2 / torch.exp(logvar)
        )

    # KL divergence
    def kullback_leibler(self, mu, logvar):
        logvar = torch.clamp(logvar, -10, 5)

        return 0.5 * torch.sum(
            mu**2 + torch.exp(logvar) -1 - logvar,
            dim=1)
        
    # Full ELBO loss (negative Evidence lower bound)
    def elbo(self, x, x_hat, mu, logvar, beta=1.0):
        recon = self.gaussian_loglikelihood(x, x_hat, logvar)
        kl = self.kullback_leibler(mu, logvar)

        return recon + beta * kl
    

In [9]:
model = MTLArchitecture(3833, 16,8)
model

MTLArchitecture(
  (encoder): Sequential(
    (0): Linear(in_features=3833, out_features=1024, bias=True)
    (1): ReLU()
    (2): Linear(in_features=1024, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=128, bias=True)
    (5): ReLU()
  )
  (mu_layer): Linear(in_features=128, out_features=16, bias=True)
  (logvar_layer): Linear(in_features=128, out_features=16, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=16, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=1024, bias=True)
    (5): ReLU()
    (6): Linear(in_features=1024, out_features=3833, bias=True)
  )
  (regression_head): Sequential(
    (0): Linear(in_features=16, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
  )
  (reg_mu): Linear(in_features=32, out_features=3, bias=True)
  (

### Downstream Tasks

tensor(3.7000)

In [14]:
def log(x):
    return np.log(2)

a = log(2)
float(np.exp(a))

2.0